In [ ]:
# =============================================================
# SP2L2 BACKTEST v4.3.6 — offline tick replay
# Uses the live-aligned setup detector; tick prices need not match M1 OHLC
# =============================================================

__author__ = "Alireza Sadabadi"
__copyright__ = "Copyright (c) 2026 Alireza Sadabadi. All rights reserved."
__credits__ = ["Alireza Sadabadi"]
__license__ = "Apache"
__version__ = "4.0"
__maintainer__ = "Alireza Sadabadi"
__email__ = "alirezasadabady@gmail.com"
__status__ = "Test"
__doc__ = "you can see the tutorials in https://youtube.com/@alirezasadabadi?si=d8o7LK_Ai1Hf68is"


import pandas as pd
import numpy as np
from Meta import *

from datetime import time, datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

SYMBOL = "XAUUSD"
TIMEFRAME = mt5.TIMEFRAME_M1
NUMBER_OF_DATA = 10000

SPIKE_CANDLE_SIZE = 1.5

PGAP_POINTS = 100
MAX_SL_DISTANCE_POINTS = 1000

INITIAL_CASH = 10000.0

# ------------------------------------------------------------
# Risk model
# ------------------------------------------------------------

RISK_PER_TRADE = 100.0

# ------------------------------------------------------------
# 1R take profit
# ------------------------------------------------------------

TP_R = 0.9

# ------------------------------------------------------------
# Second entry
# ------------------------------------------------------------

USE_SECOND_ENTRY = False

SECOND_ENTRY_VOLUME_MULTIPLIER = 2.0

# ============================================================
# OPTIONAL CANDLE COLOR FILTER
#
# If True:
#   BUY  -> all three candles must be bullish.
#   SELL -> all three candles must be bearish.
#
# If False:
#   Candle color is ignored.
#   The required higher-low / lower-high structure remains active.
# ============================================================

USE_CANDLE_COLOR_FILTER = True


# ============================================================
# OPTIONAL EMA FILTER
# ============================================================

USE_EMA_FILTER = True

EMA_PERIOD = 60


# ============================================================
# OPTIONAL EMA60 / EMA200 ALIGNMENT FILTER
#
# If True:
#   BUY  -> EMA60 > EMA200
#   SELL -> EMA60 < EMA200
#
# If False:
#   No EMA60 / EMA200 alignment restriction is applied.
# ============================================================

USE_EMA_ALIGNMENT_FILTER = False

EMA_SLOW_PERIOD = 200

# ============================================================
# REALISTIC EXECUTION MODEL
#
# IMPORTANT:
# Historical OHLC does not contain the exact intrabar tick at
# which the trigger happened. Therefore the backtest uses the
# first triggerable price (previous low/high), NOT the final
# candle low/high. A configurable spread/slippage model is then
# applied to approximate executable price.
# ============================================================

USE_REALISTIC_EXECUTION_MODEL = True
SIMULATED_SPREAD_POINTS = 30.0
SIMULATED_SLIPPAGE_POINTS = 0.0

# ============================================================
# OPTIONAL TICK DATA
# ============================================================
# If a CSV with timestamp/bid/ask is available, the engine uses
# the FIRST tick that crosses the live trigger inside each M1
# candle. This removes the major OHLC problem where the final
# candle LOW/HIGH is incorrectly used as the entry price.
#
# Accepted columns (case-insensitive):
#   timestamp,time,datetime -> tick timestamp
#   bid                   -> bid price
#   ask                   -> ask price
#
# If empty/not found, the backtest falls back to the M1 OHLC
# model and prints a warning. OHLC mode cannot reproduce live 0.5-second
# polling or determine intrabar event order; it is an estimate, not an
# interchangeable substitute for tick replay. If both SL and TP are touched
# in one candle, SL is prioritized conservatively.
# ============================================================

USE_TICK_DATA_IF_AVAILABLE = True
TICK_DATA_PATH = ""

# Timezone of naive timestamps in the tick CSV.
# The supplied XAUUSD tick sample uses the same broker/server-style
# clock as the M1 data, so the default is UTC+3.
TICK_TIMEZONE = "Etc/GMT-3"

# Live-loop execution model.
# The live trader checks the current M1 candle every 0.5 seconds.
# We model those checks on a deterministic phase within each minute.
POLL_INTERVAL_SECONDS = 0.5
POLL_PHASE_SECONDS = 0.5

# Exact live-loop replay:
# The live Trader first checks an existing pending setup. If no pending
# setup exists, it creates one and waits for the NEXT 0.5s loop before
# checking the entry. It does NOT create a setup and enter in the same loop.
EXACT_LIVE_0_5S_STATE_MACHINE = True

# When True, the forming candle is rebuilt from the supplied chronological
# Bid/Ask ticks at every 0.5s observation. This mirrors Meta.GetRates()'s
# changing current M1 candle instead of looking ahead to the candle close.
REBUILD_FORMING_CANDLE_FROM_TICKS = True

# The supplied Live Trader evaluates EMA on the current forming candle.
# Keep this separate from the older backtest FILTER_CANDLE_MODE setting.
MATCH_LIVE_FILTER_TIMING = True

# The supplied live main loop sleeps after detecting that the existing
# position has closed: sleep(60 - LOOP_SECONDS). Reproduce that cooldown.
MATCH_LIVE_POST_CLOSE_COOLDOWN = True
LIVE_POST_CLOSE_COOLDOWN_SECONDS = max(0.0, 60.0 - POLL_INTERVAL_SECONDS)

# If no local tick CSV exists, retrieve historical ticks automatically from MT5.
USE_MT5_TICK_DATA_IF_AVAILABLE = True
MT5_TICK_BATCH_SIZE = 100000
MT5_TICK_MAX_BATCHES = 50

# Optional automatic filename in the same directory as this script.
AUTO_TICK_FILENAMES = [
    "XAUUSD_ticks.csv",
    "xauusd_ticks.csv",
    "XAUUSD_tick.csv",
    "xauusd_tick.csv",
]


# ============================================================
# OPTIONAL VWAP FILTER
#
# BUY  -> execution/trigger price must be above Session VWAP
# SELL -> execution/trigger price must be below Session VWAP
# ============================================================

USE_VWAP_FILTER = False

# ============================================================
# OPTIONAL ATR / VOLATILITY REGIME FILTER
#
# Uses Risk / ATR on the entry candle's last closed bar.
# This avoids using the final close of the entry candle.
# ============================================================

USE_ATR_FILTER = False
ATR_PERIOD = 14
MIN_RISK_ATR_RATIO = 0.20
MAX_RISK_ATR_RATIO = 1.50

# ============================================================
# OPTIONAL SPREAD FILTER
#
# Requires both a maximum absolute spread and a maximum
# spread/ATR ratio. The spread itself is simulated above.
# ============================================================

USE_SPREAD_FILTER = False
MAX_SPREAD_POINTS = 30.0
MAX_SPREAD_ATR_RATIO = 0.15

# ============================================================
# OPTIONAL ADX STRENGTH FILTER
# ============================================================

USE_RANGE_FILTER = False
ADX_PERIOD = 14
MIN_ADX = 20.0

# ============================================================
# OPTIONAL ADX RISING FILTER
#
# Requires ADX at the last closed candle to be higher than the
# ADX of the candle before it.
# ============================================================

USE_ADX_RISING_FILTER = False


# ============================================================
# OPTIONAL TREND STRUCTURE FILTER
# ============================================================

USE_TREND_FILTER = True

MAX_OPPOSITE_MOVES = 1


# ============================================================
# OPTIONAL NEW YORK SESSION FILTER
#
# If True:
#   Only Entry 1 signals occurring between SESSION_START_HOUR
#   and SESSION_END_HOUR in New York time are allowed.
#
# The end hour is exclusive:
#
#   01:00 <= Entry time < 05:00
#
# If False:
#   No session/time restriction is applied.
#
# The timezone is handled with IANA timezone data so New York
# daylight-saving changes are handled automatically.
# ============================================================

USE_SESSION_FILTER = False

SESSION_START_HOUR = 1
SESSION_END_HOUR = 5

SESSION_TIMEZONE = "America/New_York"

# ============================================================
# OPTIONAL HIGH-IMPACT NEWS FILTER
#
# NEWS_EVENTS must contain UTC timestamps as strings:
# "YYYY-MM-DD HH:MM:SS"
# Add the exact high-impact event times from the economic
# calendar used for your broker/data feed.
# ============================================================

USE_NEWS_FILTER = False
NEWS_BLACKOUT_MINUTES = 30
NEWS_EVENTS = [
    # Example:
    # "2026-09-04 12:30:00",
]


# ============================================================
# MT5 INITIALIZE
# ============================================================

if not mt5.initialize():

    raise RuntimeError(
        f"MT5 initialize failed: {mt5.last_error()}"
    )


# ============================================================
# SYMBOL INFORMATION
# ============================================================

symbol_info = mt5.symbol_info(SYMBOL)

if symbol_info is None:

    mt5.shutdown()

    raise RuntimeError(
        f"Could not get symbol information for {SYMBOL}"
    )


BROKER_POINT = float(symbol_info.point)
DIGITS = int(symbol_info.digits)

if BROKER_POINT <= 0:

    mt5.shutdown()

    raise RuntimeError(
        "Invalid broker point."
    )


P_GAP_PRICE = (
    PGAP_POINTS * BROKER_POINT
)

MAX_SL_DISTANCE_PRICE = (
    MAX_SL_DISTANCE_POINTS * BROKER_POINT
)


# ============================================================
# PRINT SETTINGS
# ============================================================

print()
print("========================================")
print("BACKTEST SETTINGS")
print("========================================")

print("Symbol              :", SYMBOL)
print("Point               :", BROKER_POINT)
print("Digits              :", DIGITS)
print("Spike multiplier    :", SPIKE_CANDLE_SIZE)
print("Gap points          :", PGAP_POINTS)
print("Gap price           :", P_GAP_PRICE)
print("Max SL points       :", MAX_SL_DISTANCE_POINTS)
print("Max SL price        :", MAX_SL_DISTANCE_PRICE)
print("Initial cash        :", INITIAL_CASH)
print("Risk / trade        :", RISK_PER_TRADE)
print("TP                  :", f"{TP_R}R")
print("Second entry        :", USE_SECOND_ENTRY)
print("Candle color filter  :", USE_CANDLE_COLOR_FILTER)
print("EMA filter           :", USE_EMA_FILTER)
print("EMA period           :", EMA_PERIOD)
print("EMA alignment filter :", USE_EMA_ALIGNMENT_FILTER)
print("EMA slow period      :", EMA_SLOW_PERIOD)
print("Realistic execution  :", USE_REALISTIC_EXECUTION_MODEL)
print("Simulated spread pts :", SIMULATED_SPREAD_POINTS)
print("Slippage pts         :", SIMULATED_SLIPPAGE_POINTS)
print("VWAP filter          :", USE_VWAP_FILTER)
print("ATR filter           :", USE_ATR_FILTER)
print("ATR period           :", ATR_PERIOD)
print("Risk / ATR range     :", MIN_RISK_ATR_RATIO, "-", MAX_RISK_ATR_RATIO)
print("Spread filter        :", USE_SPREAD_FILTER)
print("Max spread points    :", MAX_SPREAD_POINTS)
print("Max spread / ATR     :", MAX_SPREAD_ATR_RATIO)
print("ADX rising filter    :", USE_ADX_RISING_FILTER)
print("News filter          :", USE_NEWS_FILTER)
print("News blackout min    :", NEWS_BLACKOUT_MINUTES)
print("Trend filter        :", USE_TREND_FILTER)
print("Range filter        :", USE_RANGE_FILTER)
print("ADX period          :", ADX_PERIOD)
print("Minimum ADX         :", MIN_ADX)
print("Session filter      :", USE_SESSION_FILTER)
print("Session timezone    :", SESSION_TIMEZONE)
print(
    "Session             :",
    f"{SESSION_START_HOUR:02d}:00 - {SESSION_END_HOUR:02d}:00"
)


# ============================================================
# GET RAW DATA
# ============================================================

# M1 rates are preferred when available, but are not mandatory when a local
# tick file is supplied. If MT5 is offline or its M1 history does not cover the
# tick period, M1 candles will be reconstructed from tick Bid prices below.
try:
    _rates = Meta.GetRates(
        symbol=SYMBOL,
        number_of_data=NUMBER_OF_DATA,
        timeFrame=TIMEFRAME
    )
    df = _rates.copy() if _rates is not None else pd.DataFrame()
except Exception:
    df = pd.DataFrame()

HAS_MT5_M1_DATA = not df.empty
if not HAS_MT5_M1_DATA:
    df = pd.DataFrame(columns=["open", "high", "low", "close"])
    df.index = pd.DatetimeIndex([], tz=TICK_TIMEZONE)

# CRITICAL TIMEZONE FIX:
# Meta.GetRates() converts MT5 rate epoch seconds with pd.to_datetime(..., unit="s"),
# which produces naive UTC timestamps. Do NOT localize those naive timestamps directly
# to the broker timezone (UTC+3): that silently shifts M1 bars by three hours relative
# to MT5 ticks, whose time_msc is explicitly converted from UTC below.
# Localize rates as UTC first, then convert to the configured broker/server timezone.
if isinstance(df.index, pd.DatetimeIndex):
    if df.index.tz is None and len(df.index):
        df.index = df.index.tz_localize("UTC").tz_convert(TICK_TIMEZONE)
    elif df.index.tz is not None:
        df.index = df.index.tz_convert(TICK_TIMEZONE)
else:
    _rate_index = pd.to_datetime(df.index, errors="raise")
    if _rate_index.tz is None:
        _rate_index = _rate_index.tz_localize("UTC")
    df.index = _rate_index.tz_convert(TICK_TIMEZONE)


# ============================================================
# NORMALIZE COLUMN NAMES
# ============================================================

df.columns = [
    str(c).lower()
    for c in df.columns
]


# ============================================================
# LOAD OPTIONAL TICK DATA
# ============================================================
#
# IMPORTANT ARCHITECTURE:
#   - Strategy/setup/filter logic is shared by both engines.
#   - Execution is NOT shared.
#   - If valid tick data is loaded, the TICK engine is used only.
#   - If no valid tick file is found, the OHLC engine is used only.
#
# The tick CSV is expected to contain full timestamps such as:
#   2026-09-23 13:00:00.059
#
# We deliberately parse naive timestamps in TICK_TIMEZONE instead
# of forcing them to UTC. This keeps them aligned with the MT5/M1
# server timestamps when the source is broker/server time.
#
# Every row is preserved. Equal timestamps are legal; _tick_seq is
# the deterministic secondary ordering key.
# ============================================================

ticks_df = None
TICK_DATA_ACTIVE = False
TICK_TIMESTAMP_COLUMN = None
TICK_SEQUENCE_COLUMN = "_tick_seq"


def _parse_tick_timestamp(series):
    """Parse tick timestamps robustly without silently losing valid rows."""
    text = series.astype("string").str.strip()

    # First try the exact format used in the supplied sample.
    parsed = pd.to_datetime(
        text,
        format="%Y-%m-%d %H:%M:%S.%f",
        errors="coerce"
    )

    # Then try timestamps without fractional seconds.
    missing = parsed.isna()
    if missing.any():
        parsed2 = pd.to_datetime(
            text.loc[missing],
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )
        parsed.loc[missing] = parsed2

    # Final fallback for unusual but still parseable rows.
    missing = parsed.isna()
    if missing.any():
        parsed2 = pd.to_datetime(
            text.loc[missing],
            format="mixed",
            errors="coerce"
        )
        parsed.loc[missing] = parsed2

    return parsed


def _prepare_tick_timezone(parsed):
    """Convert/localize parsed tick timestamps to the configured broker timezone."""
    parsed = pd.Series(parsed).reset_index(drop=True)

    if getattr(parsed.dt, "tz", None) is not None:
        return parsed.dt.tz_convert(TICK_TIMEZONE)

    return parsed.dt.tz_localize(
        TICK_TIMEZONE,
        ambiguous="NaT",
        nonexistent="NaT"
    )


def _finalize_tick_frame(raw_ticks, source_label):
    """Normalize CSV/MT5 ticks into one deterministic broker-time dataframe."""
    global ticks_df, TICK_TIMESTAMP_COLUMN, TICK_DATA_ACTIVE

    if raw_ticks is None or len(raw_ticks) == 0:
        raise ValueError("No tick rows returned")

    raw_ticks = raw_ticks.copy()
    raw_ticks.columns = [str(c).lower().strip() for c in raw_ticks.columns]
    raw_rows = len(raw_ticks)

    # MT5 supplies time_msc. Prefer it because Meta.GetTicks() also keeps
    # the original field even though its helper converts the ordinary time
    # field to seconds. This preserves sub-second ordering for 0.5s polling.
    if "time_msc" in raw_ticks.columns:
        msc = pd.to_numeric(raw_ticks["time_msc"], errors="coerce")
        parsed_timestamps = pd.to_datetime(msc, unit="ms", utc=True, errors="coerce").dt.tz_convert(TICK_TIMEZONE)
        time_col = "timestamp"
    else:
        time_col = next((c for c in ["timestamp", "time", "datetime"] if c in raw_ticks.columns), None)
        if time_col is None:
            raise ValueError("Tick data must contain timestamp/time/datetime or time_msc")
        parsed_timestamps = _prepare_tick_timezone(_parse_tick_timestamp(raw_ticks[time_col]))

    if "bid" not in raw_ticks.columns or "ask" not in raw_ticks.columns:
        raise ValueError("Tick data must contain bid and ask columns")

    bid_numeric = pd.to_numeric(raw_ticks["bid"], errors="coerce")
    ask_numeric = pd.to_numeric(raw_ticks["ask"], errors="coerce")
    raw_ticks[TICK_SEQUENCE_COLUMN] = np.arange(raw_rows, dtype=np.int64)
    raw_ticks[time_col] = parsed_timestamps
    raw_ticks["bid"] = bid_numeric
    raw_ticks["ask"] = ask_numeric

    valid_mask = (
        raw_ticks[time_col].notna()
        & raw_ticks["bid"].notna()
        & raw_ticks["ask"].notna()
    )
    raw_ticks = raw_ticks.loc[valid_mask].copy()
    if raw_ticks.empty:
        raise ValueError("No valid timestamp/bid/ask rows remain")

    raw_ticks = raw_ticks.sort_values([time_col, TICK_SEQUENCE_COLUMN], kind="mergesort")
    raw_ticks = raw_ticks.set_index(time_col)
    raw_ticks.index = pd.DatetimeIndex(raw_ticks.index)

    # M1 data and ticks must use the same broker/server timezone basis.
    if getattr(df.index, "tz", None) is None:
        df.index = pd.DatetimeIndex(df.index).tz_localize(TICK_TIMEZONE)
    else:
        df.index = pd.DatetimeIndex(df.index).tz_convert(TICK_TIMEZONE)

    ticks_df = raw_ticks
    TICK_TIMESTAMP_COLUMN = time_col
    TICK_DATA_ACTIVE = True

    # Do not require M1/tick time overlap here. M1 can be reconstructed from
    # this tick stream when MT5 history is missing or belongs to another period.


# ------------------------------------------------------------
# 1) Preferred source: local tick CSV, if explicitly supplied
#    or automatically present beside the script/current directory.
# ------------------------------------------------------------
if USE_TICK_DATA_IF_AVAILABLE:
    candidate_paths = []
    if TICK_DATA_PATH:
        candidate_paths.append(Path(TICK_DATA_PATH))
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
    else:
        script_dir = Path.cwd()
    candidate_paths.extend(script_dir / name for name in AUTO_TICK_FILENAMES)

    for candidate in candidate_paths:
        if not candidate.exists():
            continue
        try:
            with open(candidate, "r", encoding="utf-8-sig", errors="replace") as fh:
                header_line = fh.readline().strip("\r\n")
                first_data_line = fh.readline().strip("\r\n")
            header_fields = [x.strip() for x in header_line.split(",")] if header_line else []
            first_data_fields = [x.strip() for x in first_data_line.split(",")] if first_data_line else []
            if (
                len(header_fields) == 3
                and [x.lower() for x in header_fields] == ["timestamp", "bid", "ask"]
                and len(first_data_fields) >= 5
            ):
                tick_columns = ["Timestamp", "bid", "ask", "bid_volume", "ask_volume"]
                raw_ticks = pd.read_csv(candidate, skiprows=1, header=None,
                                        names=tick_columns, usecols=range(5),
                                        dtype=str, engine="python")
            else:
                raw_ticks = pd.read_csv(candidate, dtype=str, engine="python")
            _finalize_tick_frame(raw_ticks, f"CSV: {candidate}")
            break
        except Exception as exc:
            print("Tick CSV load failed         :", candidate)
            print("Reason                      :", exc)
            TICK_DATA_ACTIVE = False
            ticks_df = None


# ------------------------------------------------------------
# 2) Automatic MT5 fallback.
# IMPORTANT: USE_TICK_DATA_IF_AVAILABLE is the master switch. When False,
# neither local CSV ticks nor MT5 historical ticks may be loaded.
# USE_MT5_TICK_DATA_IF_AVAILABLE only controls fallback when the master
# switch is True and no usable local tick CSV was found.
# ------------------------------------------------------------
if USE_TICK_DATA_IF_AVAILABLE and (not TICK_DATA_ACTIVE) and USE_MT5_TICK_DATA_IF_AVAILABLE:
    try:
        m1_start = pd.Timestamp(df.index.min())
        m1_end = pd.Timestamp(df.index.max()) + pd.Timedelta(minutes=1)
        if m1_start.tzinfo is None:
            m1_start = m1_start.tz_localize(TICK_TIMEZONE)
        if m1_end.tzinfo is None:
            m1_end = m1_end.tz_localize(TICK_TIMEZONE)

        cursor = m1_start.tz_convert("UTC").to_pydatetime()
        end_utc = m1_end.tz_convert("UTC")
        batches = []
        total_rows = 0
        last_tick_time = None

        for batch_no in range(MT5_TICK_MAX_BATCHES):
            batch = Meta.GetTicks(
                symbol=SYMBOL,
                number_of_data=MT5_TICK_BATCH_SIZE,
                fromDate=cursor
            )
            if batch is None or len(batch) == 0:
                break

            batch = batch.copy()
            batches.append(batch)
            total_rows += len(batch)

            if "time_msc" in batch.columns:
                last_msc = pd.to_numeric(batch["time_msc"], errors="coerce").dropna()
                if len(last_msc):
                    last_tick_time = pd.to_datetime(int(last_msc.iloc[-1]), unit="ms", utc=True)
            else:
                bidx = pd.DatetimeIndex(batch.index)
                if len(bidx):
                    last_tick_time = bidx[-1]
                    if last_tick_time.tzinfo is None:
                        last_tick_time = last_tick_time.tz_localize("UTC")
                    else:
                        last_tick_time = last_tick_time.tz_convert("UTC")

            if last_tick_time is None or last_tick_time >= end_utc:
                break

            # Meta.GetTicks ultimately calls copy_ticks_from(), which is
            # inclusive. Advance by 1 ms to avoid repeating the boundary tick.
            cursor = (last_tick_time + pd.Timedelta(milliseconds=1)).to_pydatetime()
            if len(batch) < MT5_TICK_BATCH_SIZE:
                break

        if batches:
            raw_ticks = pd.concat(batches, axis=0, ignore_index=True)
            # Restrict to the M1 interval before normalization.
            if "time_msc" in raw_ticks.columns:
                ts_utc = pd.to_datetime(
                    pd.to_numeric(raw_ticks["time_msc"], errors="coerce"),
                    unit="ms", utc=True, errors="coerce"
                )
                mask = (ts_utc >= m1_start.tz_convert("UTC")) & (ts_utc < end_utc)
                raw_ticks = raw_ticks.loc[mask].copy()
            _finalize_tick_frame(raw_ticks, "MT5 Meta.GetTicks")
        else:
            pass
    except Exception as exc:
        TICK_DATA_ACTIVE = False
        ticks_df = None
        print("MT5 tick retrieval failed   :", exc)

if not USE_TICK_DATA_IF_AVAILABLE:
    print("Tick data                   : DISABLED BY USE_TICK_DATA_IF_AVAILABLE=False")
    print("Local CSV tick loading      : SKIPPED")
    print("MT5 historical tick loading : SKIPPED")
    print("Tick execution engine       : INACTIVE")
    print("Entry/exit engine           : M1 OHLC")
elif not TICK_DATA_ACTIVE:
    print("Tick data                   : NOT AVAILABLE")
    print("Tick execution engine       : INACTIVE")
    print("Entry/exit engine           : M1 OHLC")


# If MT5 M1 rates are absent or cover a different period than the local ticks,
# build M1 candles from the tick Bid stream. If the periods partly overlap,
# retain broker M1 bars where present and fill missing minutes from tick candles.
if TICK_DATA_ACTIVE:
    _tick_bid_m1 = ticks_df["bid"].astype(float).resample("1min").ohlc().dropna()
    _tick_bid_m1 = _tick_bid_m1.rename(columns={
        "open": "open", "high": "high", "low": "low", "close": "close"
    })
    _tick_bid_m1.index = pd.DatetimeIndex(_tick_bid_m1.index).tz_convert(TICK_TIMEZONE)

    if df.empty:
        df = _tick_bid_m1.copy()
    else:
        # Keep the source M1 values where available; use tick-built bars only
        # for missing timestamps, including historical dates outside MT5 range.
        df = df.combine_first(_tick_bid_m1)

    # Tick replay can only trade where tick observations exist. Restrict the
    # signal timeline to the actual tick interval to avoid signals outside it.
    _tick_first_minute = ticks_df.index.min().floor("min")
    _tick_last_minute = ticks_df.index.max().floor("min")
    df = df.loc[(df.index >= _tick_first_minute) & (df.index <= _tick_last_minute)]

if df.empty:
    raise RuntimeError(
        "No usable market data. Provide a readable local tick CSV, or connect MT5 and make sure M1 history is available."
    )


def _localize_to_tick_timezone(value):
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        return ts.tz_localize(TICK_TIMEZONE)
    return ts.tz_convert(TICK_TIMEZONE)


def _tick_slice(start, end):
    """Fast timezone-aware slice preserving duplicate timestamps and source order."""
    if not TICK_DATA_ACTIVE:
        return None

    start = _localize_to_tick_timezone(start)
    end = _localize_to_tick_timezone(end)

    idx = ticks_df.index
    left = idx.searchsorted(start, side="left")
    right = idx.searchsorted(end, side="left")
    return ticks_df.iloc[left:right]


def get_ticks_for_candle(candle_time, start_time=None):
    if not TICK_DATA_ACTIVE:
        return None

    candle_start = _localize_to_tick_timezone(candle_time)
    candle_end = candle_start + pd.Timedelta(minutes=1)

    if start_time is not None:
        start = _localize_to_tick_timezone(start_time)
        if start < candle_start:
            start = candle_start
    else:
        start = candle_start

    return _tick_slice(start, candle_end)


def find_first_polling_trigger(candle_time, direction, previous_extreme):
    """Fast deterministic replay of the live robot's 0.5-second polling."""
    t = get_ticks_for_candle(candle_time)
    if t is None or len(t) == 0:
        return None

    candle_start = _localize_to_tick_timezone(candle_time)
    candle_end = candle_start + pd.Timedelta(minutes=1)
    first_poll = candle_start + pd.Timedelta(seconds=POLL_PHASE_SECONDS)
    interval = pd.Timedelta(seconds=POLL_INTERVAL_SECONDS)
    if first_poll >= candle_end:
        return None

    # The live trader only needs the running Bid LOW/HIGH.  Compute cumulative
    # extrema once, then test all poll instants with vectorized searchsorted.
    tick_times = t.index
    bid = t["bid"].to_numpy(dtype=float, copy=False)
    if bid.size == 0:
        return None

    if direction == "BUY":
        running_extreme = np.minimum.accumulate(bid)
        hit_condition = running_extreme < float(previous_extreme)
    else:
        running_extreme = np.maximum.accumulate(bid)
        hit_condition = running_extreme > float(previous_extreme)

    poll_count = int(np.ceil((candle_end - first_poll) / interval))
    poll_offsets = np.arange(poll_count, dtype=np.int64)
    poll_times = first_poll + pd.to_timedelta(poll_offsets * POLL_INTERVAL_SECONDS, unit="s")
    poll_times = poll_times[poll_times < candle_end]
    if len(poll_times) == 0:
        return None

    rights = tick_times.searchsorted(poll_times, side="right")
    valid = rights > 0
    if not valid.any():
        return None

    check_rights = rights[valid] - 1
    hit = hit_condition[check_rights]
    if not hit.any():
        return None

    first_idx = int(np.flatnonzero(valid)[np.flatnonzero(hit)[0]])
    tick_pos = int(rights[first_idx] - 1)
    poll_time = poll_times[first_idx]
    trigger_price = float(running_extreme[tick_pos])

    # Execute at the latest quote visible at the poll, not at the cumulative
    # intrabar extreme. This is the closest deterministic replay of a market
    # order sent after the live strategy's polling check.
    quote_pos = int(rights[first_idx] - 1)
    if direction == "BUY":
        entry = float(t.iloc[quote_pos]["ask"])
    else:
        entry = float(t.iloc[quote_pos]["bid"])

    return poll_time, trigger_price, entry

def get_ticks_for_candle_from(candle_time, start_time=None):
    """Tick slice for a candle, optionally starting at a precise timestamp."""
    return get_ticks_for_candle(candle_time, start_time=start_time)

# Tick execution intentionally does not require Bid ranges to match M1 OHLC.
# M1 candles drive signal detection; the tick stream independently drives fills,
# SL/TP touches, and tick-replay execution. Only valid tick data and time overlap
# are required to activate the tick engine.


def find_tick_second_entry_event(
    candle_time,
    direction,
    second_level,
    state,
    start_time=None,
):
    """
    Tick-aware Entry 2 with mandatory pullback + re-cross confirmation.

    BUY:
      1) Bid must first fall to/below second_level (arm).
      2) Only after arming, Bid must rise back above second_level.
      3) Execute BUY at Ask on that re-cross tick.

    SELL:
      1) Ask must first rise to/above second_level (arm).
      2) Only after arming, Ask must fall back below second_level.
      3) Execute SELL at Bid on that re-cross tick.

    The function mutates only the supplied state dict and returns
    (event_time, trigger_price, executable_entry) or None.
    """
    t = get_ticks_for_candle_from(candle_time, start_time=start_time)
    if t is None or len(t) == 0:
        return None

    level = float(second_level)
    armed = bool(state.get("armed", False))

    for tick_time, tick in t.iterrows():
        bid = float(tick["bid"])
        ask = float(tick["ask"])

        if direction == "BUY":
            if not armed:
                if bid <= level:
                    armed = True
                    state["armed"] = True
                continue

            if bid > level:
                state["armed"] = True
                return tick_time, bid, ask

        else:
            if not armed:
                if ask >= level:
                    armed = True
                    state["armed"] = True
                continue

            if ask < level:
                state["armed"] = True
                return tick_time, ask, bid

    state["armed"] = armed
    return None


def find_tick_second_entry_exit(
    candle_time,
    direction,
    sl,
    tp,
    entry_time,
):
    """Return the first SL/TP event for Entry 2 after its actual entry tick."""
    return find_first_tick_exit(
        candle_time,
        direction,
        sl,
        tp,
        entry_time=entry_time,
    )


def find_first_tick_exit(candle_time, direction, sl, tp, entry_time=None):
    """Return first SL/TP event after entry using executable Bid/Ask."""
    t = get_ticks_for_candle(candle_time, start_time=entry_time)
    if t is None or len(t) == 0:
        return None

    if entry_time is not None:
        entry_ts = _localize_to_tick_timezone(entry_time)
        # Strictly AFTER entry. Equal timestamp is not considered an exit.
        t = t.loc[t.index > entry_ts]
        if t.empty:
            return None

    if direction == "BUY":
        # BUY closes against Bid.
        for tick_time, tick in t.iterrows():
            bid = float(tick["bid"])
            if bid <= float(sl):
                return float(sl), tick_time, "SL"
            if bid >= float(tp):
                return float(tp), tick_time, "TP"

    else:
        # SELL closes against Ask.
        for tick_time, tick in t.iterrows():
            ask = float(tick["ask"])
            if ask >= float(sl):
                return float(sl), tick_time, "SL"
            if ask <= float(tp):
                return float(tp), tick_time, "TP"

    return None


required = [
    "open",
    "high",
    "low",
    "close"
]


for col in required:

    if col not in df.columns:

        mt5.shutdown()

        raise RuntimeError(
            f"Missing required column: {col}\n"
            f"Available columns: {list(df.columns)}"
        )


# ============================================================
# DATETIME INDEX
# ============================================================

if not isinstance(df.index, pd.DatetimeIndex):

    possible_time_columns = [
        "time",
        "datetime",
        "date",
        "local time"
    ]

    found_time = None

    for c in possible_time_columns:

        if c in df.columns:

            found_time = c
            break

    if found_time is not None:

        df[found_time] = pd.to_datetime(
            df[found_time]
        )

        df.set_index(
            found_time,
            inplace=True
        )


# ============================================================
# SORT
# ============================================================

df.sort_index(
    inplace=True
)


# ============================================================
# FORCE NUMERIC OHLC
# ============================================================

for col in required:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


df.dropna(
    subset=required,
    inplace=True
)


# ============================================================
# CREATE SIGNAL COLUMNS
# ============================================================

df["position"] = 0

df["sl_buy"] = np.nan
df["sl_sell"] = np.nan

df["entry_buy"] = np.nan
df["entry_sell"] = np.nan

df["entry_buy_2x"] = np.nan
df["entry_sell_2x"] = np.nan

df["spike_index"] = pd.Series(
    index=df.index,
    dtype="object"
)

df["spike_body"] = np.nan
df["sl_distance"] = np.nan
df["trigger_price"] = np.nan


# ============================================================
# EMA CALCULATION
# ============================================================

df["EMA"] = (
    df["close"]
    .ewm(
        span=EMA_PERIOD,
        adjust=False
    )
    .mean()
)


df["EMA200"] = (
    df["close"]
    .ewm(
        span=EMA_SLOW_PERIOD,
        adjust=False
    )
    .mean()
)


# ============================================================
# ATR / ADX CALCULATION
# ============================================================

def calculate_atr(data, period):

    high = data["high"]
    low = data["low"]
    close = data["close"]
    previous_close = close.shift(1)

    tr = pd.concat(
        [
            high - low,
            (high - previous_close).abs(),
            (low - previous_close).abs()
        ],
        axis=1
    ).max(axis=1)

    return tr.ewm(
        alpha=1 / period,
        adjust=False
    ).mean()


def calculate_adx(data, period):

    high = data["high"]
    low = data["low"]
    close = data["close"]

    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = pd.Series(
        np.where(
            (up_move > down_move) & (up_move > 0),
            up_move,
            0.0
        ),
        index=data.index
    )

    minus_dm = pd.Series(
        np.where(
            (down_move > up_move) & (down_move > 0),
            down_move,
            0.0
        ),
        index=data.index
    )

    previous_close = close.shift(1)

    tr = pd.concat(
        [
            high - low,
            (high - previous_close).abs(),
            (low - previous_close).abs()
        ],
        axis=1
    ).max(axis=1)

    atr = tr.ewm(
        alpha=1 / period,
        adjust=False
    ).mean()

    plus_di = (
        100 *
        plus_dm.ewm(
            alpha=1 / period,
            adjust=False
        ).mean() / atr
    )

    minus_di = (
        100 *
        minus_dm.ewm(
            alpha=1 / period,
            adjust=False
        ).mean() / atr
    )

    denominator = plus_di + minus_di

    dx = (
        100 *
        (plus_di - minus_di).abs() /
        denominator.replace(0, np.nan)
    )

    return dx.ewm(
        alpha=1 / period,
        adjust=False
    ).mean()


df["ATR"] = calculate_atr(df, ATR_PERIOD)
df["ADX"] = calculate_adx(df, ADX_PERIOD)


# ============================================================
# SESSION VWAP
#
# Uses tick volume when available. If tick volume is unavailable,
# equal volume is used rather than silently failing. The session
# resets by New York calendar date.
# ============================================================

if "tick_volume" in df.columns:
    vwap_volume = pd.to_numeric(
        df["tick_volume"],
        errors="coerce"
    ).fillna(1.0)
else:
    vwap_volume = pd.Series(
        1.0,
        index=df.index
    )

typical_price = (
    df["high"] + df["low"] + df["close"]
) / 3.0

ny_index = pd.DatetimeIndex(df.index)
if ny_index.tz is None:
    ny_index_aware = ny_index.tz_localize("Etc/GMT-3")
else:
    ny_index_aware = ny_index.tz_convert("UTC")

ny_dates = ny_index_aware.tz_convert(SESSION_TIMEZONE).date

volume_price = typical_price * vwap_volume
df["VWAP"] = (
    volume_price.groupby(ny_dates).cumsum()
    /
    vwap_volume.groupby(ny_dates).cumsum().replace(0, np.nan)
)

# ============================================================
# NEW YORK SESSION HELPER
# ============================================================

NEW_YORK_TZ = ZoneInfo(
    SESSION_TIMEZONE
)


def is_in_new_york_session(timestamp):
    """
    Returns True when timestamp falls inside:

        SESSION_START_HOUR <= time < SESSION_END_HOUR

    in New York local time.

    Naive timestamps are interpreted as UTC+3, matching the
    timestamp convention used by the supplied XAUUSD data.
    """

    if pd.isna(timestamp):

        return False

    ts = pd.Timestamp(timestamp)

    if ts.tzinfo is None:

        # The supplied dataset timestamps are treated as UTC+3.
        ts = ts.tz_localize(
            "Etc/GMT-3"
        )

    else:

        ts = ts.tz_convert(
            "UTC"
        )

    ny_time = ts.tz_convert(
        NEW_YORK_TZ
    ).time()

    start_time = time(
        SESSION_START_HOUR,
        0
    )

    end_time = time(
        SESSION_END_HOUR,
        0
    )

    return (
        start_time
        <=
        ny_time
        <
        end_time
    )


# ============================================================
# NEWS FILTER HELPER
# ============================================================

NEWS_EVENT_TIMES = []

for event in NEWS_EVENTS:
    try:
        ts = pd.Timestamp(event, tz="UTC")
        NEWS_EVENT_TIMES.append(ts)
    except Exception:
        print("Invalid NEWS_EVENTS timestamp skipped:", event)


def is_near_high_impact_news(timestamp):

    if not USE_NEWS_FILTER:
        return True

    ts = pd.Timestamp(timestamp)

    if ts.tzinfo is None:
        ts = ts.tz_localize("Etc/GMT-3").tz_convert("UTC")
    else:
        ts = ts.tz_convert("UTC")

    window = pd.Timedelta(minutes=NEWS_BLACKOUT_MINUTES)

    for event_ts in NEWS_EVENT_TIMES:
        if abs(ts - event_ts) <= window:
            return False

    return True


# ============================================================
# REFERENCE CANDLES
# ============================================================

for lag in [1, 2, 3]:

    df[f"open_{lag}"] = (
        df["open"].shift(lag)
    )

    df[f"high_{lag}"] = (
        df["high"].shift(lag)
    )

    df[f"low_{lag}"] = (
        df["low"].shift(lag)
    )

    df[f"close_{lag}"] = (
        df["close"].shift(lag)
    )


# ============================================================
# REMOVE FIRST 3 INVALID ROWS
# ============================================================

df = df.dropna(
    subset=[
        "open_1",
        "high_1",
        "low_1",
        "close_1",
        "open_2",
        "high_2",
        "low_2",
        "close_2",
        "open_3",
        "high_3",
        "low_3",
        "close_3"
    ]
).copy()


# ============================================================
# SETUP CONDITIONS
# ============================================================

buy_setup = (

    (
        df["close_1"]
        >
        df["close_2"]
    )

    &

    (
        df["open_1"]
        >
        df["open_2"]
    )

    &

    (
        df["close_2"]
        >
        df["close_3"]
    )

    &

    (
        df["open_2"]
        >
        df["open_3"]
    )

    &

    (
        df["low_1"]
        >
        df["low_2"]
    )

    &

    (
        df["low_2"]
        >
        df["low_3"]
    )

    &

    (
        df["low_1"]
        >
        df["high_3"] + P_GAP_PRICE
    )

)


buy_color_condition = True

if USE_CANDLE_COLOR_FILTER:

    buy_color_condition = (

        (df["close_1"] > df["open_1"])

        &

        (df["close_2"] > df["open_2"])

        &

        (df["close_3"] > df["open_3"])
    )


buy_setup = (
    buy_setup
    &
    buy_color_condition
)


buy_spike_body = (
    df["close_2"]
    -
    df["open_2"]
)

buy_before_body = (
    df["close_3"]
    -
    df["open_3"]
)

buy_after_body = (
    df["close_1"]
    -
    df["open_1"]
)


buy_spike = (

    buy_spike_body
    >
    SPIKE_CANDLE_SIZE
    *
    buy_before_body

) & (

    buy_spike_body
    >
    SPIKE_CANDLE_SIZE
    *
    buy_after_body

)


buy_setup = (
    buy_setup
    &
    buy_spike
)


# ============================================================
# SELL SETUP
# ============================================================

sell_setup = (

    (
        df["close_1"]
        <
        df["close_2"]
    )

    &

    (
        df["open_1"]
        <
        df["open_2"]
    )

    &

    (
        df["close_2"]
        <
        df["close_3"]
    )

    &

    (
        df["open_2"]
        <
        df["open_3"]
    )

    &

    (
        df["high_1"]
        <
        df["high_2"]
    )

    &

    (
        df["high_2"]
        <
        df["high_3"]
    )

    &

    (
        df["high_1"]
        <
        df["low_3"] - P_GAP_PRICE
    )

)


sell_color_condition = True

if USE_CANDLE_COLOR_FILTER:

    sell_color_condition = (

        (df["close_1"] < df["open_1"])

        &

        (df["close_2"] < df["open_2"])

        &

        (df["close_3"] < df["open_3"])
    )


sell_setup = (
    sell_setup
    &
    sell_color_condition
)


sell_spike_body = (
    df["open_2"]
    -
    df["close_2"]
)

sell_before_body = (
    df["open_3"]
    -
    df["close_3"]
)

sell_after_body = (
    df["open_1"]
    -
    df["close_1"]
)


sell_spike = (

    sell_spike_body
    >
    SPIKE_CANDLE_SIZE
    *
    sell_before_body

) & (

    sell_spike_body
    >
    SPIKE_CANDLE_SIZE
    *
    sell_after_body

)


sell_setup = (
    sell_setup
    &
    sell_spike
)


# ============================================================
# LIVE-ALIGNED EXECUTION / STOP-LOSS SETTINGS
# ============================================================
#
# This engine intentionally models the live SP2L workflow:
#
#   current candle = -1
#   after-spike    = -2
#   spike candle   = -3
#   before-spike   = -4
#
# A setup is detected while the current candle is forming.
# If the current candle already crosses the trigger, entry can
# happen on that SAME candle.
#
# Because M1 OHLC does not contain tick order, this is an
# approximation of live execution, not an exact tick replay.
# ============================================================

USE_LIVE_ALIGNED_EXECUTION = True

# This file implements the live-aligned state machine only. Fail loudly if a
# user disables this legacy flag rather than silently running the same engine.
if not USE_LIVE_ALIGNED_EXECUTION:
    raise ValueError(
        "USE_LIVE_ALIGNED_EXECUTION=False is not supported by v4.3.3; "
        "this version implements the live-aligned engine only."
    )

# Entry model:
#   False (recommended) -> use the previous candle's breakout threshold as
#                          the trigger reference and apply adverse execution
#                          costs. Never use the final candle LOW/HIGH as if
#                          it were the known live fill price.
#   True is retained only for legacy comparison and is NOT recommended.
USE_CURRENT_EXTREME_AS_ENTRY = False

# Allow setup and trigger to occur on the same candle.
ALLOW_SAME_CANDLE_SETUP_ENTRY = True

# If a new setup appears while another setup is pending,
# replace the old pending setup.
REPLACE_PENDING_ON_NEW_SETUP = True

# Trend filter:
#   True  -> include the entry/trigger candle
#   False -> exclude it
TREND_INCLUDES_ENTRY_CANDLE = True

# Indicator filter timing:
#   "CURRENT"  -> use the forming/trigger candle
#   "PREVIOUS" -> use the last closed candle
FILTER_CANDLE_MODE = "PREVIOUS"

# ------------------------------------------------------------
# STOP-LOSS MODE
#
# "BEFORE_SPIKE":
#   BUY  -> LOW of the candle immediately before the spike
#   SELL -> HIGH of the candle immediately before the spike
#
# "LOWER_LOW":
#   BUY  -> nearest LowerLow found by walking backward from
#           the candle immediately before the spike.
#
# "LOWER_HIGH":
#   SELL -> nearest LowerHigh (technically a higher-high
#           swing for the short-side stop) found by walking
#           backward from the candle immediately before spike.
#
# The requested user-facing choices are:
#   BEFORE_SPIKE
#   LOWER_LOW
#
# For SELL, LOWER_LOW automatically maps to the corresponding
# upper swing: LOWER_HIGH.
# ------------------------------------------------------------

SL_MODE = "BEFORE_SPIKE"

# Maximum number of candles to search backward for a swing stop.
LOWER_LOW_LOOKBACK = 100

# ------------------------------------------------------------
# Optional execution spread/slippage for OHLC-only fallback.
# M1 OHLC obtained from MT5 is treated as BID OHLC. Therefore:
#   BUY entry executes at Ask ~= trigger Bid + full spread + slippage
#   SELL entry executes at Bid ~= trigger Bid - slippage
# Tick mode uses observed Bid/Ask quotes directly and does not use these
# synthetic execution adjustments for the entry.
# ------------------------------------------------------------

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _is_finite(value):
    return value is not None and np.isfinite(float(value))


def get_stop_loss(setup_pos, direction):
    """
    setup_pos is the current candle on which the live setup is
    recognized.

    Therefore:
        setup_pos - 1 = after-spike
        setup_pos - 2 = spike
        setup_pos - 3 = before-spike
    """

    before_spike_pos = setup_pos - 3

    if before_spike_pos < 1:
        return None, None

    if SL_MODE == "BEFORE_SPIKE":

        if direction == "BUY":
            sl = float(df.iloc[before_spike_pos]["low"])
        else:
            sl = float(df.iloc[before_spike_pos]["high"])

        return sl, before_spike_pos

    # --------------------------------------------------------
    # BUY: nearest LowerLow
    # --------------------------------------------------------

    if direction == "BUY":

        for pos in range(
            before_spike_pos,
            max(0, before_spike_pos - LOWER_LOW_LOOKBACK),
            -1
        ):

            if pos <= 0 or pos >= len(df) - 1:
                continue

            candidate = float(df.iloc[pos]["low"])
            left_low = float(df.iloc[pos - 1]["low"])
            right_low = float(df.iloc[pos + 1]["low"])

            if candidate < left_low and candidate < right_low:
                return candidate, pos

        return None, None

    # --------------------------------------------------------
    # SELL: corresponding upper swing (higher-high)
    # --------------------------------------------------------

    for pos in range(
        before_spike_pos,
        max(0, before_spike_pos - LOWER_LOW_LOOKBACK),
        -1
    ):

        if pos <= 0 or pos >= len(df) - 1:
            continue

        candidate = float(df.iloc[pos]["high"])
        left_high = float(df.iloc[pos - 1]["high"])
        right_high = float(df.iloc[pos + 1]["high"])

        if candidate > left_high and candidate > right_high:
            return candidate, pos

    return None, None


def trend_is_valid(start_pos, entry_pos, direction):
    if not USE_TREND_FILTER:
        return True

    stop_pos = (
        entry_pos + 1
        if TREND_INCLUDES_ENTRY_CANDLE
        else entry_pos
    )

    consecutive_opposite = 0

    for pos in range(start_pos + 1, stop_pos):

        if direction == "BUY":

            current_high = float(df.iloc[pos]["high"])
            previous_high = float(df.iloc[pos - 1]["high"])

            if current_high > previous_high:
                consecutive_opposite = 0
            else:
                consecutive_opposite += 1

        else:

            current_low = float(df.iloc[pos]["low"])
            previous_low = float(df.iloc[pos - 1]["low"])

            if current_low < previous_low:
                consecutive_opposite = 0
            else:
                consecutive_opposite += 1

        if consecutive_opposite > MAX_OPPOSITE_MOVES:
            return False

    return True


def entry_filters_are_valid_live(
    entry_pos,
    direction,
    execution_entry,
    risk
):
    """
    Live-aligned indicator filter evaluation.

    CURRENT:
        use entry_pos itself.

    PREVIOUS:
        use entry_pos - 1.

    Note:
    VWAP/session/news are evaluated at the trigger candle time.
    """

    if FILTER_CANDLE_MODE.upper() == "CURRENT":
        filter_pos = entry_pos
    else:
        filter_pos = entry_pos - 1

    if filter_pos < 0:
        return False

    entry_idx = df.index[entry_pos]
    filter_row = df.iloc[filter_pos]

    # EMA
    if USE_EMA_FILTER:

        entry_close = float(filter_row["close"])
        entry_ema = float(filter_row["EMA"])

        if direction == "BUY" and entry_close <= entry_ema:
            return False

        if direction == "SELL" and entry_close >= entry_ema:
            return False

    # EMA60 / EMA200 alignment
    if USE_EMA_ALIGNMENT_FILTER:

        ema60 = float(filter_row["EMA"])
        ema200 = float(filter_row["EMA200"])

        if direction == "BUY" and ema60 <= ema200:
            return False

        if direction == "SELL" and ema60 >= ema200:
            return False

    # ADX strength
    if USE_RANGE_FILTER:

        entry_adx = float(filter_row["ADX"])

        if not np.isfinite(entry_adx) or entry_adx < MIN_ADX:
            return False

    # ADX rising
    if USE_ADX_RISING_FILTER:

        previous_filter_pos = filter_pos - 1

        if previous_filter_pos < 0:
            return False

        adx_now = float(df.iloc[filter_pos]["ADX"])
        adx_previous = float(
            df.iloc[previous_filter_pos]["ADX"]
        )

        if (
            not np.isfinite(adx_now)
            or not np.isfinite(adx_previous)
            or adx_now <= adx_previous
        ):
            return False

    # ATR regime
    if USE_ATR_FILTER:

        atr = float(filter_row["ATR"])

        if not np.isfinite(atr) or atr <= 0:
            return False

        risk_atr_ratio = risk / atr

        if (
            risk_atr_ratio < MIN_RISK_ATR_RATIO
            or
            risk_atr_ratio > MAX_RISK_ATR_RATIO
        ):
            return False

    # VWAP
    if USE_VWAP_FILTER:

        vwap = float(filter_row["VWAP"])

        if not np.isfinite(vwap):
            return False

        if direction == "BUY" and execution_entry <= vwap:
            return False

        if direction == "SELL" and execution_entry >= vwap:
            return False

    # Spread
    if USE_SPREAD_FILTER:

        spread_points = SIMULATED_SPREAD_POINTS

        if spread_points > MAX_SPREAD_POINTS:
            return False

        atr = float(filter_row["ATR"])

        if not np.isfinite(atr) or atr <= 0:
            return False

        spread_price = spread_points * BROKER_POINT
        spread_atr_ratio = spread_price / atr

        if spread_atr_ratio > MAX_SPREAD_ATR_RATIO:
            return False

    # Session
    if USE_SESSION_FILTER:

        if not is_in_new_york_session(entry_idx):
            return False

    # News
    if USE_NEWS_FILTER:

        if not is_near_high_impact_news(entry_idx):
            return False

    return True


def calculate_execution_price(trigger_price, direction):
    """Approximate a market fill from BID-based M1 OHLC data.

    This function is used only by the OHLC fallback. Tick mode uses actual
    Bid/Ask observations at the simulated polling instant.
    """
    trigger_price = float(trigger_price)
    if not USE_REALISTIC_EXECUTION_MODEL:
        return trigger_price

    spread_price = max(0.0, float(SIMULATED_SPREAD_POINTS)) * BROKER_POINT
    slippage_price = max(0.0, float(SIMULATED_SLIPPAGE_POINTS)) * BROKER_POINT

    if direction == "BUY":
        # OHLC is assumed to be Bid; a long market order buys at Ask.
        return trigger_price + spread_price + slippage_price

    # A short market order sells at Bid; adverse slippage lowers the fill.
    return trigger_price - slippage_price


def setup_exists_at_pos(pos):
    """
    Re-evaluate the original 3-reference-candle setup at the
    current candle. This mirrors the live trader's:
        -1 current
        -2 after-spike
        -3 spike
        -4 before-spike
    """

    if pos < 3:
        return False, False

    current = df.iloc[pos]
    after = df.iloc[pos - 1]
    spike = df.iloc[pos - 2]
    before = df.iloc[pos - 3]

    # BUY structure + live trigger condition
    # The live trader only creates a pending BUY setup when the
    # CURRENT forming candle has already made a lower low than
    # the immediately previous candle.
    buy_trigger = (
        float(current["low"]) < float(after["low"])
    )

    buy_structure = (
        after["close"] > spike["close"]
        and after["open"] > spike["open"]
        and spike["close"] > before["close"]
        and spike["open"] > before["open"]
        and after["low"] > spike["low"]
        and spike["low"] > before["low"]
        and after["low"] > before["high"] + P_GAP_PRICE
    )

    buy_color = True
    if USE_CANDLE_COLOR_FILTER:
        buy_color = (
            after["close"] > after["open"]
            and spike["close"] > spike["open"]
            and before["close"] > before["open"]
        )

    buy_spike_body = (
        float(spike["close"]) - float(spike["open"])
    )
    buy_before_body = (
        float(before["close"]) - float(before["open"])
    )
    buy_after_body = (
        float(after["close"]) - float(after["open"])
    )

    buy_spike_ok = (
        buy_spike_body
        >
        SPIKE_CANDLE_SIZE * buy_before_body
        and
        buy_spike_body
        >
        SPIKE_CANDLE_SIZE * buy_after_body
    )

    buy_setup = bool(
        buy_trigger
        and buy_structure
        and buy_color
        and buy_spike_ok
    )

    # SELL structure + live trigger condition
    # The live trader only creates a pending SELL setup when the
    # CURRENT forming candle has already made a higher high than
    # the immediately previous candle.
    sell_trigger = (
        float(current["high"]) > float(after["high"])
    )

    sell_structure = (
        after["close"] < spike["close"]
        and after["open"] < spike["open"]
        and spike["close"] < before["close"]
        and spike["open"] < before["open"]
        and after["high"] < spike["high"]
        and spike["high"] < before["high"]
        and after["high"] < before["low"] - P_GAP_PRICE
    )

    sell_color = True
    if USE_CANDLE_COLOR_FILTER:
        sell_color = (
            after["close"] < after["open"]
            and spike["close"] < spike["open"]
            and before["close"] < before["open"]
        )

    sell_spike_body = (
        float(spike["open"]) - float(spike["close"])
    )
    sell_before_body = (
        float(before["open"]) - float(before["close"])
    )
    sell_after_body = (
        float(after["open"]) - float(after["close"])
    )

    sell_spike_ok = (
        sell_spike_body
        >
        SPIKE_CANDLE_SIZE * sell_before_body
        and
        sell_spike_body
        >
        SPIKE_CANDLE_SIZE * sell_after_body
    )

    sell_setup = bool(
        sell_trigger
        and sell_structure
        and sell_color
        and sell_spike_ok
    )

    return buy_setup, sell_setup


def create_pending_setup(pos, direction):
    sl, sl_pos = get_stop_loss(pos, direction)

    if sl is None:
        return None

    spike_pos = pos - 2

    spike_body = abs(
        float(df.iloc[spike_pos]["close"])
        -
        float(df.iloc[spike_pos]["open"])
    )

    return {
        "setup_pos": pos,
        "setup_time": df.index[pos],
        "direction": direction,
        "sl": float(sl),
        "sl_pos": int(sl_pos),
        "spike_pos": int(spike_pos),
        "spike_time": df.index[spike_pos],
        "spike_body": spike_body,
    }


def try_activate_pending_ohlc(pending, entry_pos):
    """OHLC-only execution engine. Used only when tick engine is inactive."""
    if pending is None:
        return None

    direction = pending["direction"]
    sl = float(pending["sl"])
    current = df.iloc[entry_pos]
    previous = df.iloc[entry_pos - 1]

    if direction == "BUY":
        if float(current["low"]) >= float(previous["low"]):
            return None
        # The breakout threshold is knowable; the final candle low is not
        # knowable at the instant the live order would be sent.
        trigger_price = (
            float(current["low"])
            if USE_CURRENT_EXTREME_AS_ENTRY
            else float(previous["low"])
        )
        entry = (
            trigger_price
            if USE_CURRENT_EXTREME_AS_ENTRY
            else calculate_execution_price(trigger_price, "BUY")
        )
        entry_time = df.index[entry_pos]
        risk = entry - sl

    else:
        if float(current["high"]) <= float(previous["high"]):
            return None
        trigger_price = (
            float(current["high"])
            if USE_CURRENT_EXTREME_AS_ENTRY
            else float(previous["high"])
        )
        entry = (
            trigger_price
            if USE_CURRENT_EXTREME_AS_ENTRY
            else calculate_execution_price(trigger_price, "SELL")
        )
        entry_time = df.index[entry_pos]
        risk = sl - entry

    return _build_activation(pending, entry_pos, entry, trigger_price, entry_time, risk)


def try_activate_pending_tick(pending, entry_pos):
    """Tick-backed execution engine that simulates the live 0.5s polling loop."""
    if pending is None or not TICK_DATA_ACTIVE:
        return None

    direction = pending["direction"]
    sl = float(pending["sl"])
    previous = df.iloc[entry_pos]
    trigger_extreme = float(
        df.iloc[entry_pos - 1]["low"]
        if direction == "BUY"
        else df.iloc[entry_pos - 1]["high"]
    )

    tick_event = find_first_polling_trigger(
        df.index[entry_pos],
        direction,
        trigger_extreme
    )

    # Critical: with tick mode active, no tick event means NO ENTRY.
    if tick_event is None:
        return None

    entry_time, trigger_price, entry = tick_event

    if direction == "BUY":
        risk = entry - sl
    else:
        risk = sl - entry

    return _build_activation(
        pending,
        entry_pos,
        entry,
        trigger_price,
        entry_time,
        risk
    )


def _build_activation(
    pending,
    entry_pos,
    entry,
    trigger_price,
    entry_time,
    risk
):
    """Shared post-execution validation and position construction."""
    if risk <= 0:
        return None

    if risk > MAX_SL_DISTANCE_PRICE:
        return None

    if not trend_is_valid(
        pending["setup_pos"],
        entry_pos,
        pending["direction"]
    ):
        return None

    if not entry_filters_are_valid_live(
        entry_pos,
        pending["direction"],
        entry,
        risk
    ):
        return None

    direction = pending["direction"]

    tp = (
        entry + TP_R * risk
        if direction == "BUY"
        else entry - TP_R * risk
    )

    second_entry = (
        entry - risk / 2.0
        if direction == "BUY"
        else entry + risk / 2.0
    )

    return {
        "setup_id": pending["setup_pos"],
        "signal_time": df.index[entry_pos],
        "setup_time": pending["setup_time"],
        "direction": direction,
        "entry": entry,
        "trigger_price": trigger_price,
        "sl": float(pending["sl"]),
        "tp": tp,
        "risk": risk,
        "base_risk": risk,
        "second_entry": second_entry,
        "second_entry_active": False,
        "second_entry_armed": False,
        "second_entry_entry_time": None,
        "second_entry_scan_time": entry_time,
        "spike_body": pending["spike_body"],
        "spike_time": pending["spike_time"],
        "sl_pos": pending["sl_pos"],
        "legs": [
            {
                "setup_id": pending["setup_pos"],
                "entry_number": 1,
                "entry_time": entry_time,
                "activation_time": entry_time,
                "direction": direction,
                "entry": entry,
                "sl": float(pending["sl"]),
                "tp": tp,
                "risk": risk,
                "base_risk": risk,
                "volume_multiplier": 1.0,
                "spike_body": pending["spike_body"],
                "setup_time": pending["setup_time"],
                "trigger_price": trigger_price,
            }
        ],
    }


# ============================================================
# SIGNAL COLUMNS
# ============================================================

# Reset / recreate output columns.
df["position"] = 0
df["sl_buy"] = np.nan
df["sl_sell"] = np.nan
df["entry_buy"] = np.nan
df["entry_sell"] = np.nan
df["entry_buy_2x"] = np.nan
df["entry_sell_2x"] = np.nan
df["spike_index"] = pd.Series(index=df.index, dtype="object")
df["spike_body"] = np.nan
df["sl_distance"] = np.nan
df["trigger_price"] = np.nan
df["setup_time"] = pd.Series(index=df.index, dtype="object")
df["sl_mode"] = SL_MODE


# ============================================================
# OHLC BACKTEST
# ============================================================

trades = []
equity = INITIAL_CASH
pending_setup = None
active_setup = None
cooldown_until = None


def close_leg(
    leg,
    exit_price,
    exit_time,
    exit_reason
):

    global equity

    direction = leg["direction"]
    entry = leg["entry"]
    sl = leg["sl"]
    tp = leg["tp"]
    risk = leg["risk"]
    base_risk = leg["base_risk"]
    volume_multiplier = leg["volume_multiplier"]

    if direction == "BUY":
        price_r = (exit_price - entry) / risk
    else:
        price_r = (entry - exit_price) / risk

    r_multiple = (
        price_r
        * volume_multiplier
        * (risk / base_risk)
    )

    pnl = r_multiple * RISK_PER_TRADE
    equity += pnl

    engine_audit["closed_trade_legs"] += 1
    trades.append({
        "setup_id": leg["setup_id"],
        "entry_number": leg["entry_number"],
        "setup_time": leg.get("setup_time"),
        "entry_time": leg["entry_time"],
        "activation_time": leg["activation_time"],
        "exit_time": exit_time,
        "direction": direction,
        "entry": entry,
        "sl": sl,
        "tp": tp,
        "risk": risk,
        "base_risk": base_risk,
        "volume_multiplier": volume_multiplier,
        "exit": exit_price,
        "R": r_multiple,
        "PnL": pnl,
        "exit_reason": exit_reason,
        "spike_body": leg["spike_body"],
        "sl_points": base_risk / BROKER_POINT,
        "trigger_price": leg.get("trigger_price"),
        "sl_mode": SL_MODE,
    })


# ============================================================
# EXACT LIVE 0.5-SECOND STATE-MACHINE REPLAY
# ============================================================
#
# The previous engine searched the entire forming candle for the first
# polling trigger. That is NOT identical to the live Trader. In the live
# code, every 0.5s loop executes in this order:
#
#   1) If a position exists -> manage it.
#   2) If a pending setup exists -> check_pending_buy/sell().
#      If valid, enter NOW.
#   3) Otherwise detect_buy_setup/detect_sell_setup().
#      If valid, CREATE pending setup only; entry is checked on the NEXT
#      0.5s loop.
#
# This section replays that state machine chronologically from ticks.
# Trigger state uses the forming Bid candle, while market entry uses the
# executable quote at that poll (Ask for BUY, Bid for SELL). TP is derived
# from that entry and the fixed SL, mirroring the post-fill live calculation.
# ============================================================

trades = []
equity = INITIAL_CASH
pending_setup = None
active_setup = None


def close_leg(
    leg,
    exit_price,
    exit_time,
    exit_reason
):
    global equity

    direction = leg["direction"]
    entry = float(leg["entry"])
    sl = float(leg["sl"])
    tp = float(leg["tp"])
    risk = float(leg["risk"])
    base_risk = float(leg["base_risk"])
    volume_multiplier = float(leg["volume_multiplier"])

    if direction == "BUY":
        price_r = (float(exit_price) - entry) / risk
    else:
        price_r = (entry - float(exit_price)) / risk

    r_multiple = price_r * volume_multiplier * (risk / base_risk)
    pnl = r_multiple * RISK_PER_TRADE
    equity += pnl

    engine_audit["closed_trade_legs"] += 1
    trades.append({
        "setup_id": leg["setup_id"],
        "entry_number": leg["entry_number"],
        "setup_time": leg.get("setup_time"),
        "entry_time": leg["entry_time"],
        "activation_time": leg["activation_time"],
        "exit_time": exit_time,
        "direction": direction,
        "entry": entry,
        "sl": sl,
        "tp": tp,
        "risk": risk,
        "base_risk": base_risk,
        "volume_multiplier": volume_multiplier,
        "exit": float(exit_price),
        "R": r_multiple,
        "PnL": pnl,
        "exit_reason": exit_reason,
        "spike_body": leg["spike_body"],
        "sl_points": base_risk / BROKER_POINT,
        "trigger_price": leg.get("trigger_price"),
        "sl_mode": SL_MODE,
    })


def _live_setup_from_snapshot(pos, direction, snapshot):
    """Exact create_pending_buy/create_pending_sell equivalent."""
    # pos is the current forming M1 candle. -4 from the live data is
    # pos-3 in the full dataframe: before-spike.
    before_pos = pos - 3
    spike_pos = pos - 2
    if before_pos < 1:
        return None

    if direction == "BUY":
        sl = float(df.iloc[before_pos]["low"])
    else:
        sl = float(df.iloc[before_pos]["high"])

    spike = df.iloc[spike_pos]
    spike_body = abs(float(spike["close"]) - float(spike["open"]))

    return {
        "setup_pos": int(pos),
        "setup_time": df.index[pos],
        "direction": direction,
        "sl": sl,
        "sl_pos": int(before_pos),
        "spike_pos": int(spike_pos),
        "spike_time": df.index[spike_pos],
        "spike_body": spike_body,
    }


def _live_setup_snapshot(pos, current):
    """Exact detect_buy_setup/detect_sell_setup using a forming snapshot."""
    if pos < 4:
        return False, False

    after = df.iloc[pos - 1]
    spike = df.iloc[pos - 2]
    before = df.iloc[pos - 3]

    c_open = float(current["open"])
    c_high = float(current["high"])
    c_low = float(current["low"])
    c_close = float(current["close"])

    buy0 = c_low < float(after["low"])
    buy1 = float(after["close"]) > float(spike["close"])
    buy2 = float(after["open"]) > float(spike["open"])
    buy3 = float(spike["close"]) > float(before["close"])
    buy4 = float(spike["open"]) > float(before["open"])
    buy5 = float(after["close"]) > float(after["open"])
    buy6 = float(spike["close"]) > float(spike["open"])
    buy7 = float(before["close"]) > float(before["open"])
    buy_gap = float(after["low"]) > float(before["high"]) + P_GAP_PRICE

    spike_buy_body = float(spike["close"]) - float(spike["open"])
    before_buy_body = float(before["close"]) - float(before["open"])
    after_buy_body = float(after["close"]) - float(after["open"])
    buy_spike = (
        spike_buy_body > SPIKE_CANDLE_SIZE * before_buy_body
        and spike_buy_body > SPIKE_CANDLE_SIZE * after_buy_body
    )

    buy = (
        buy0 and buy1 and buy2 and buy3 and buy4 and
        buy5 and buy6 and buy7 and buy_gap and buy_spike
    )

    sell0 = c_high > float(after["high"])
    sell1 = float(after["close"]) < float(spike["close"])
    sell2 = float(after["open"]) < float(spike["open"])
    sell3 = float(spike["close"]) < float(before["close"])
    sell4 = float(spike["open"]) < float(before["open"])
    sell5 = float(after["close"]) < float(after["open"])
    sell6 = float(spike["close"]) < float(spike["open"])
    sell7 = float(before["close"]) < float(before["open"])
    sell_gap = float(after["high"]) < float(before["low"]) - P_GAP_PRICE

    spike_sell_body = float(spike["open"]) - float(spike["close"])
    before_sell_body = float(before["open"]) - float(before["close"])
    after_sell_body = float(after["open"]) - float(after["close"])
    sell_spike = (
        spike_sell_body > SPIKE_CANDLE_SIZE * before_sell_body
        and spike_sell_body > SPIKE_CANDLE_SIZE * after_sell_body
    )

    sell = (
        sell0 and sell1 and sell2 and sell3 and sell4 and
        sell5 and sell6 and sell7 and sell_gap and sell_spike
    )

    return bool(buy), bool(sell)


def _live_trend_valid_with_snapshot(start_pos, entry_pos, direction, current):
    if not USE_TREND_FILTER:
        return True

    consecutive = 0
    for pos in range(start_pos + 1, entry_pos + 1):
        if pos == entry_pos:
            cur_high = float(current["high"])
            cur_low = float(current["low"])
            prev_high = float(df.iloc[pos - 1]["high"])
            prev_low = float(df.iloc[pos - 1]["low"])
        else:
            cur_high = float(df.iloc[pos]["high"])
            cur_low = float(df.iloc[pos]["low"])
            prev_high = float(df.iloc[pos - 1]["high"])
            prev_low = float(df.iloc[pos - 1]["low"])

        if direction == "BUY":
            if cur_high > prev_high:
                consecutive = 0
            else:
                consecutive += 1
        else:
            if cur_low < prev_low:
                consecutive = 0
            else:
                consecutive += 1

        if consecutive > MAX_OPPOSITE_MOVES:
            return False

    return True


def _live_entry_filters_snapshot(pos, direction, current, ema_value):
    """Exact filter semantics of the supplied live Trader."""
    entry_idx = df.index[pos]

    # The supplied live Trader uses the current forming candle for EMA.
    if USE_EMA_FILTER:
        entry_close = float(current["close"])
        entry_ema = float(ema_value)
        if not np.isfinite(entry_ema):
            return False
        if direction == "BUY" and entry_close <= entry_ema:
            return False
        if direction == "SELL" and entry_close >= entry_ema:
            return False

    if USE_RANGE_FILTER:
        adx = float(df.iloc[pos]["ADX"])
        if not np.isfinite(adx) or adx < MIN_ADX:
            return False

    if USE_ADX_RISING_FILTER:
        if pos < 2:
            return False
        a = float(df.iloc[pos - 1]["ADX"])
        b = float(df.iloc[pos - 2]["ADX"])
        if not np.isfinite(a) or not np.isfinite(b) or a <= b:
            return False

    if USE_SESSION_FILTER and not is_in_new_york_session(entry_idx):
        return False

    if USE_NEWS_FILTER and not is_near_high_impact_news(entry_idx):
        return False

    return True


def _activate_live_pending(pos, pending, current, ema_value, poll_time):
    """Exact check_pending_buy/check_pending_sell + TP calculation."""
    if (
        not ALLOW_SAME_CANDLE_SETUP_ENTRY
        and int(pos) == int(pending.get("setup_pos", -1))
    ):
        return None

    direction = pending["direction"]
    sl = float(pending["sl"])

    if direction == "BUY":
        current_extreme = float(current["low"])
        previous_extreme = float(df.iloc[pos - 1]["low"])
        if current_extreme >= previous_extreme:
            return None
        # BUY trigger is observed from BID candle low, but the market order
        # executes at ASK. The quote at this poll, not the running LOW, is
        # the fill reference (Meta.SendOrder buys at Ask).
        entry = float(current.get("ask", current.get("close")))
        risk = entry - sl
    else:
        current_extreme = float(current["high"])
        previous_extreme = float(df.iloc[pos - 1]["high"])
        if current_extreme <= previous_extreme:
            return None
        # SELL market orders execute at BID. Keep the running HIGH only as
        # the trigger/extreme diagnostic, not as the fill price.
        entry = float(current.get("bid", current.get("close")))
        risk = sl - entry

    if risk <= 0:
        if "engine_audit" in globals():
            engine_audit["invalid_risk_nonpositive"] += 1
            examples = engine_audit.setdefault("invalid_risk_examples", [])
            if len(examples) < 8:
                examples.append({
                    "reason": "risk_nonpositive",
                    "time": str(poll_time),
                    "direction": direction,
                    "setup_pos": int(pending.get("setup_pos", -1)),
                    "entry": float(entry),
                    "sl": float(sl),
                    "risk": float(risk),
                })
        return "INVALID"

    if risk > MAX_SL_DISTANCE_PRICE:
        if "engine_audit" in globals():
            engine_audit["invalid_risk_too_large"] += 1
            examples = engine_audit.setdefault("invalid_risk_examples", [])
            if len(examples) < 8:
                examples.append({
                    "reason": "risk_exceeds_max",
                    "time": str(poll_time),
                    "direction": direction,
                    "setup_pos": int(pending.get("setup_pos", -1)),
                    "entry": float(entry),
                    "sl": float(sl),
                    "risk": float(risk),
                    "max_risk": float(MAX_SL_DISTANCE_PRICE),
                })
        return "INVALID"

    if not _live_trend_valid_with_snapshot(
        pending["setup_pos"], pos, direction, current
    ):
        return None

    if not _live_entry_filters_snapshot(pos, direction, current, ema_value):
        return None

    tp = entry + TP_R * risk if direction == "BUY" else entry - TP_R * risk

    return {
        "setup_id": pending["setup_pos"],
        "signal_time": df.index[pos],
        "setup_time": pending["setup_time"],
        "direction": direction,
        "entry": entry,
        "trigger_price": current_extreme,
        "sl": sl,
        "tp": tp,
        "risk": risk,
        "base_risk": risk,
        "second_entry": entry - risk / 2.0 if direction == "BUY" else entry + risk / 2.0,
        "second_entry_active": False,
        "second_entry_armed": False,
        "spike_body": pending["spike_body"],
        "spike_time": pending["spike_time"],
        "sl_pos": pending["sl_pos"],
        "legs": [{
            "setup_id": pending["setup_pos"],
            "entry_number": 1,
            "entry_time": poll_time,
            "activation_time": poll_time,
            "direction": direction,
            "entry": entry,
            "sl": sl,
            "tp": tp,
            "risk": risk,
            "base_risk": risk,
            "volume_multiplier": 1.0,
            "spike_body": pending["spike_body"],
            "setup_time": pending["setup_time"],
            "trigger_price": current_extreme,
        }],
    }


def _ticks_for_replay_candle(candle_time):
    return get_ticks_for_candle(candle_time)


# Engine audit counters. These describe execution-state events, not PnL.
engine_audit = {
    "polls_or_bars_checked": 0,
    "invalid_setup_positions_blocked": 0,
    "setup_detection_hits": 0,
    "pending_created": 0,
    "pending_replaced": 0,
    "invalid_pending_cleared": 0,
    "invalid_risk_nonpositive": 0,
    "invalid_risk_too_large": 0,
    "invalid_risk_examples": [],
    "entry_setups_activated": 0,
    "closed_trade_legs": 0,
}

# Prevent an invalid pending from being recreated repeatedly on the same
# forming candle after its SL/entry geometry has already failed validation.
rejected_setup_positions = set()

# We intentionally use the chronological tick stream as the clock.
# If ticks are unavailable, retain the old OHLC engine below.
if TICK_DATA_ACTIVE and EXACT_LIVE_0_5S_STATE_MACHINE:
    for i in range(4, len(df)):
        candle_time = df.index[i]
        t = _ticks_for_replay_candle(candle_time)
        if t is None or t.empty:
            continue

        # Only replay candles that actually have tick coverage.
        candle_start = _localize_to_tick_timezone(candle_time)
        candle_end = candle_start + pd.Timedelta(minutes=1)
        t = t[(t.index >= candle_start) & (t.index < candle_end)]
        if t.empty:
            continue

        first_poll = candle_start + pd.Timedelta(seconds=POLL_PHASE_SECONDS)
        poll_interval = pd.Timedelta(seconds=POLL_INTERVAL_SECONDS)
        poll_time = first_poll
        tick_times = t.index
        tick_pos = 0
        running_high = -np.inf
        running_low = np.inf
        last_bid = None
        last_ask = None

        # Previous closed candle values are from M1, matching the live
        # strategy's cached historical data.
        while poll_time < candle_end:
            engine_audit["polls_or_bars_checked"] += 1
            # Match the live main loop's post-close sleep. During this
            # interval Strategy() is not called and any pending setup is
            # already cleared.
            if (
                MATCH_LIVE_POST_CLOSE_COOLDOWN
                and cooldown_until is not None
                and poll_time < cooldown_until
            ):
                poll_time += poll_interval
                continue

            right = tick_times.searchsorted(poll_time, side="right")
            while tick_pos < right:
                tick = t.iloc[tick_pos]
                bid = float(tick["bid"])
                ask = float(tick["ask"])
                if np.isfinite(bid):
                    running_low = min(running_low, bid)
                    running_high = max(running_high, bid)
                    last_bid = bid
                if np.isfinite(ask):
                    last_ask = ask
                tick_pos += 1

            if last_bid is None:
                poll_time += poll_interval
                continue

            # Reconstruct the forming candle visible to Meta.GetRates().
            base = df.iloc[i]
            current = {
                "open": float(base["open"]),
                "high": float(running_high),
                "low": float(running_low),
                "close": float(last_bid),
                "bid": float(last_bid),
                "ask": float(last_ask) if last_ask is not None else float(last_bid),
            }

            # EMA update exactly as the supplied live lightweight updater.
            prev_ema = float(df.iloc[i - 1]["EMA"])
            alpha = 2.0 / (EMA_PERIOD + 1.0)
            ema_value = alpha * current["close"] + (1.0 - alpha) * prev_ema

            # 1) Existing position/pending setup is handled BEFORE setup
            # detection, exactly like Strategy().
            if active_setup is not None:
                direction = active_setup["direction"]

                # ----------------------------------------------------
                # MANAGE EACH OPEN LEG INDEPENDENTLY
                # ----------------------------------------------------
                # Entry 1 and Entry 2 have their own state.  They share
                # the original SL/TP levels, but each leg can be closed
                # independently and therefore cannot cause a duplicate
                # Entry 2 order.
                remaining_legs = []
                legs_to_close = []

                for leg in active_setup["legs"]:
                    entry_ts = _localize_to_tick_timezone(leg["entry_time"])
                    exit_ticks = t[(t.index > entry_ts) & (t.index <= poll_time)]
                    exit_price = None
                    exit_reason = None
                    exit_ts = None

                    for tick_time, tick in exit_ticks.iterrows():
                        bid = float(tick["bid"])
                        ask = float(tick["ask"])
                        if direction == "BUY":
                            if bid <= float(leg["sl"]):
                                # Stop is a market execution after the quote crosses SL.
                                exit_price, exit_reason, exit_ts = bid, "SL", tick_time
                                break
                            if bid >= float(leg["tp"]):
                                exit_price, exit_reason, exit_ts = float(leg["tp"]), "TP", tick_time
                                break
                        else:
                            if ask >= float(leg["sl"]):
                                # Stop is a market execution after the quote crosses SL.
                                exit_price, exit_reason, exit_ts = ask, "SL", tick_time
                                break
                            if ask <= float(leg["tp"]):
                                exit_price, exit_reason, exit_ts = float(leg["tp"]), "TP", tick_time
                                break

                    if exit_price is None:
                        remaining_legs.append(leg)
                    else:
                        legs_to_close.append((leg, exit_price, exit_ts, exit_reason))

                for leg, exit_price, exit_ts, exit_reason in legs_to_close:
                    close_leg(leg, exit_price, exit_ts, exit_reason)

                active_setup["legs"] = remaining_legs

                # If every open leg is closed, the setup is finished.
                if not active_setup["legs"]:
                    active_setup = None
                    pending_setup = None
                    if MATCH_LIVE_POST_CLOSE_COOLDOWN:
                        cooldown_until = poll_time + pd.Timedelta(
                            seconds=LIVE_POST_CLOSE_COOLDOWN_SECONDS
                        )
                    poll_time += poll_interval
                    continue

                # ----------------------------------------------------
                # ENTRY 2 — MANDATORY PULLBACK + RE-CROSS
                # ----------------------------------------------------
                # Entry 2 is armed only after price reaches the calculated
                # second-entry level. It is NOT opened on the first touch.
                # After arming, price must cross back through the level.
                # The Entry 2 TP is EXACTLY Entry 1's TP.
                if (
                    USE_SECOND_ENTRY
                    and not active_setup["second_entry_active"]
                ):
                    scan_from = active_setup.get("second_entry_scan_time")
                    second_state = {"armed": active_setup["second_entry_armed"]}
                    event = find_tick_second_entry_event(
                        candle_time,
                        direction,
                        active_setup["second_entry"],
                        second_state,
                        start_time=scan_from,
                    )
                    active_setup["second_entry_armed"] = bool(second_state.get("armed", False))

                    if event is not None:
                        event_time, trigger_price, second_entry_price = event
                        second_risk = (
                            second_entry_price - active_setup["sl"]
                            if direction == "BUY"
                            else active_setup["sl"] - second_entry_price
                        )

                        if second_risk > 0 and second_risk <= MAX_SL_DISTANCE_PRICE:
                            second_tp = active_setup["tp"]
                            second_leg = {
                                "setup_id": active_setup["setup_id"],
                                "entry_number": 2,
                                "entry_time": event_time,
                                "activation_time": event_time,
                                "direction": direction,
                                "entry": float(second_entry_price),
                                "sl": float(active_setup["sl"]),
                                "tp": float(second_tp),
                                "risk": float(second_risk),
                                "base_risk": float(active_setup["base_risk"]),
                                "volume_multiplier": float(SECOND_ENTRY_VOLUME_MULTIPLIER),
                                "spike_body": active_setup["spike_body"],
                                "setup_time": active_setup["setup_time"],
                                "trigger_price": float(trigger_price),
                            }
                            active_setup["legs"].append(second_leg)
                            active_setup["second_entry_active"] = True
                            active_setup["second_entry_entry_time"] = event_time
                            active_setup["second_entry_armed"] = True
                            active_setup["second_entry_scan_time"] = event_time
                        else:
                            # Invalid second-entry risk: do not repeatedly
                            # try to open it on later polls.
                            active_setup["second_entry_active"] = True
                    else:
                        active_setup["second_entry_armed"] = bool(
                            active_setup.get("second_entry_armed", False)
                        )
                        active_setup["second_entry_scan_time"] = poll_time

                poll_time += poll_interval
                continue

            entered_this_poll = False
            if pending_setup is not None:
                activated = _activate_live_pending(
                    i, pending_setup, current, ema_value, poll_time
                )
                if activated == "INVALID":
                    # Reject this setup position for the remainder of this M1
                    # candle. Without this guard, setup detection immediately
                    # recreated the same impossible pending every 0.5 seconds.
                    rejected_pos = int(pending_setup.get("setup_pos", -1))
                    if rejected_pos >= 0 and rejected_pos not in rejected_setup_positions:
                        rejected_setup_positions.add(rejected_pos)
                        engine_audit["invalid_setup_positions_blocked"] += 1
                    pending_setup = None
                    engine_audit["invalid_pending_cleared"] += 1
                elif activated is not None:
                    engine_audit["entry_setups_activated"] += 1
                    active_setup = activated
                    pending_setup = None
                    entered_this_poll = True
                    if activated["direction"] == "BUY":
                        df.iloc[i, df.columns.get_loc("position")] = 1
                        df.iloc[i, df.columns.get_loc("entry_buy")] = activated["entry"]
                        df.iloc[i, df.columns.get_loc("sl_buy")] = activated["sl"]
                    else:
                        df.iloc[i, df.columns.get_loc("position")] = -1
                        df.iloc[i, df.columns.get_loc("entry_sell")] = activated["entry"]
                        df.iloc[i, df.columns.get_loc("sl_sell")] = activated["sl"]
                    df.iloc[i, df.columns.get_loc("sl_distance")] = activated["risk"]
                    df.iloc[i, df.columns.get_loc("trigger_price")] = activated["trigger_price"]
                    df.iloc[i, df.columns.get_loc("spike_body")] = activated["spike_body"]
                    df.iloc[i, df.columns.get_loc("spike_index")] = activated["spike_time"]
                    df.iloc[i, df.columns.get_loc("setup_time")] = activated["setup_time"]

            # Match the actual live Strategy flow: when pending activation
            # returns None, Strategy falls through to new-setup detection.
            # Therefore a fresh setup may replace the pending one when the
            # configured option is enabled. The prior version printed this
            # setting but never used it in either execution engine.
            same_candle_wait = (
                pending_setup is not None
                and not ALLOW_SAME_CANDLE_SETUP_ENTRY
                and i == int(pending_setup.get("setup_pos", -1))
            )
            if (
                not entered_this_poll
                and not same_candle_wait
                and i not in rejected_setup_positions
            ):
                if pending_setup is None or REPLACE_PENDING_ON_NEW_SETUP:
                    buy_now, sell_now = _live_setup_snapshot(i, current)
                    if buy_now or sell_now:
                        engine_audit["setup_detection_hits"] += 1
                        new_direction = "BUY" if buy_now else "SELL"
                        new_pending = _live_setup_from_snapshot(i, new_direction, current)
                        if new_pending is not None:
                            if pending_setup is None:
                                engine_audit["pending_created"] += 1
                            else:
                                engine_audit["pending_replaced"] += 1
                            pending_setup = new_pending

            poll_time += poll_interval

else:
    # Fallback: original candle state machine when tick data is unavailable.
    for i in range(len(df)):
        engine_audit["polls_or_bars_checked"] += 1
        idx = df.index[i]
        row = df.iloc[i]
        candle_low = float(row["low"])
        candle_high = float(row["high"])

        if active_setup is not None:
            direction = active_setup["direction"]
            sl = active_setup["sl"]
            tp = active_setup["tp"]
            exit_price = None
            exit_reason = None
            spread_price = (
                max(0.0, float(SIMULATED_SPREAD_POINTS)) * BROKER_POINT
                if USE_REALISTIC_EXECUTION_MODEL else 0.0
            )
            candle_open = float(row["open"])
            if direction == "BUY":
                # Long exits are triggered on BID. If the candle opens through
                # SL, use the open as the adverse fill; otherwise use the SL.
                if candle_open <= sl:
                    exit_price, exit_reason = candle_open, "SL_GAP"
                elif candle_low <= sl:
                    exit_price, exit_reason = sl, "SL"
                elif candle_high >= tp:
                    exit_price, exit_reason = tp, "TP"
            else:
                # Short exits are triggered on ASK. OHLC is assumed BID-based,
                # so translate high/low/open to synthetic ASK using spread.
                ask_open = candle_open + spread_price
                ask_high = candle_high + spread_price
                ask_low = candle_low + spread_price
                if ask_open >= sl:
                    exit_price, exit_reason = ask_open, "SL_GAP"
                elif ask_high >= sl:
                    exit_price, exit_reason = sl, "SL"
                elif ask_low <= tp:
                    exit_price, exit_reason = tp, "TP"
            if exit_price is not None:
                for leg in active_setup["legs"]:
                    close_leg(leg, exit_price, idx, exit_reason)
                active_setup = None
                continue
            continue

        if pending_setup is not None:
            activated = try_activate_pending_ohlc(pending_setup, i)
            if activated is not None:
                engine_audit["entry_setups_activated"] += 1
                active_setup = activated
                pending_setup = None
                if activated["direction"] == "BUY":
                    df.iloc[i, df.columns.get_loc("position")] = 1
                    df.iloc[i, df.columns.get_loc("entry_buy")] = activated["entry"]
                    df.iloc[i, df.columns.get_loc("sl_buy")] = activated["sl"]
                else:
                    df.iloc[i, df.columns.get_loc("position")] = -1
                    df.iloc[i, df.columns.get_loc("entry_sell")] = activated["entry"]
                    df.iloc[i, df.columns.get_loc("sl_sell")] = activated["sl"]
                df.iloc[i, df.columns.get_loc("sl_distance")] = activated["risk"]
                df.iloc[i, df.columns.get_loc("trigger_price")] = activated["trigger_price"]
                df.iloc[i, df.columns.get_loc("spike_body")] = activated["spike_body"]
                df.iloc[i, df.columns.get_loc("spike_index")] = activated["spike_time"]
                df.iloc[i, df.columns.get_loc("setup_time")] = activated["setup_time"]
                continue

            # Keep the pending setup only when replacement is disabled.
            # When enabled, re-evaluate setup detection on this bar rather
            # than silently blocking every later setup behind a stale one.
            if not REPLACE_PENDING_ON_NEW_SETUP:
                continue

        if i >= 4:
            # Same structural/setup predicate as Tick mode, evaluated once
            # at the completed OHLC candle. This cannot reproduce intrabar
            # 0.5-second timing, but it avoids silently using a different
            # legacy setup definition in the fallback engine.
            current_snapshot = {
                "open": float(row["open"]),
                "high": float(row["high"]),
                "low": float(row["low"]),
                "close": float(row["close"]),
                "bid": float(row["close"]),
                "ask": float(row["close"]) + (
                    max(0.0, float(SIMULATED_SPREAD_POINTS)) * BROKER_POINT
                    if USE_REALISTIC_EXECUTION_MODEL else 0.0
                ),
            }
            buy_now, sell_now = _live_setup_snapshot(i, current_snapshot)
            if buy_now or sell_now:
                engine_audit["setup_detection_hits"] += 1
                new_direction = "BUY" if buy_now else "SELL"
                new_pending = _live_setup_from_snapshot(i, new_direction, current_snapshot)
                if new_pending is not None:
                    if pending_setup is None:
                        engine_audit["pending_created"] += 1
                    else:
                        engine_audit["pending_replaced"] += 1
                    pending_setup = new_pending

# ============================================================
# CLOSE REMAINING OPEN POSITION
# ============================================================

if active_setup is not None:

    last_idx = df.index[-1]
    last_close = float(df.iloc[-1]["close"])

    for leg in active_setup["legs"]:
        close_leg(
            leg,
            last_close,
            last_idx,
            "END_OF_DATA"
        )

    active_setup = None


# ============================================================
# TRADES DATAFRAME
# ============================================================

trades_df = pd.DataFrame(trades)


# ============================================================
# STATISTICS
# ============================================================

if len(trades_df) > 0:

    wins = trades_df["R"] > 0
    losses = trades_df["R"] <= 0

    win_count = int(wins.sum())
    loss_count = int(losses.sum())
    total_trades = len(trades_df)

    win_rate = (
        win_count / total_trades * 100
    )

    total_R = trades_df["R"].sum()
    total_pnl = trades_df["PnL"].sum()

    entry1_trades = trades_df[
        trades_df["entry_number"] == 1
    ]

    entry2_trades = trades_df[
        trades_df["entry_number"] == 2
    ]

    entry2_activations = len(entry2_trades)
    entry2_wins = int((entry2_trades["R"] > 0).sum())
    entry2_losses = int((entry2_trades["R"] <= 0).sum())
    entry2_R = entry2_trades["R"].sum()

    if entry2_activations > 0:
        entry2_win_rate = (
            entry2_wins / entry2_activations * 100
        )
    else:
        entry2_win_rate = 0.0

    setup_R = (
        trades_df
        .groupby("setup_id")["R"]
        .sum()
    )

    setup_count = len(setup_R)
    setup_wins = int((setup_R > 0).sum())

    setup_win_rate = (
        setup_wins / setup_count * 100
        if setup_count > 0
        else 0.0
    )

    print()
    print("========================================")
    print("ENTRY 1 / ENTRY 2")
    print("========================================")

    print("Entry 1 trades   :", len(entry1_trades))
    print("Entry 1 Total R  :", round(entry1_trades["R"].sum(), 2))
    print("Entry 2 active   :", entry2_activations)
    print("Entry 2 wins     :", entry2_wins)
    print("Entry 2 losses   :", entry2_losses)
    print("Entry 2 win rate :", round(entry2_win_rate, 2), "%")
    print("Entry 2 Total R  :", round(entry2_R, 2))
    print("Setups           :", setup_count)
    print("Setup win rate   :", round(setup_win_rate, 2), "%")

    buy_trades = trades_df[
        trades_df["direction"] == "BUY"
    ]
    sell_trades = trades_df[
        trades_df["direction"] == "SELL"
    ]

    buy_wins = int((buy_trades["R"] > 0).sum())
    sell_wins = int((sell_trades["R"] > 0).sum())

    buy_R = buy_trades["R"].sum()
    sell_R = sell_trades["R"].sum()

    gross_profit = trades_df.loc[
        trades_df["PnL"] > 0, "PnL"
    ].sum()

    gross_loss = abs(
        trades_df.loc[
            trades_df["PnL"] < 0, "PnL"
        ].sum()
    )

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss > 0
        else np.inf
    )

    cumulative_equity = (
        INITIAL_CASH
        +
        trades_df["PnL"].cumsum()
    )

    equity_peak = cumulative_equity.cummax()
    drawdown = cumulative_equity - equity_peak
    max_drawdown = drawdown.min()

    print()
    print("========================================")
    print(" BACKTEST")
    print("========================================")

    print("Completed trade legs:", total_trades)
    print("Unique traded setups:", int(trades_df["setup_id"].nunique()) if "setup_id" in trades_df else total_trades)
    print("Execution engine:", "TICK / POLL REPLAY" if TICK_DATA_ACTIVE and EXACT_LIVE_0_5S_STATE_MACHINE else "OHLC FALLBACK (APPROXIMATE)")
    print("OHLC entry uses current extreme:", bool(USE_CURRENT_EXTREME_AS_ENTRY))
    if not (TICK_DATA_ACTIVE and EXACT_LIVE_0_5S_STATE_MACHINE):
        print("OHLC caveat: intrabar order is unknown; if SL and TP are both touched, the engine prioritizes SL.")

    print()
    print("========================================")
    print("TRADE STATISTICS")
    print("========================================")

    print("Wins       :", win_count)
    print("Losses     :", loss_count)
    print("Win rate   :", round(win_rate, 2), "%")
    print("Total R    :", round(total_R, 2))
    print("Total PnL  :", round(total_pnl, 2))
    print("Final cash :", round(equity, 2))
    print(
        "Return     :",
        round(total_pnl / INITIAL_CASH * 100, 2),
        "%"
    )
    print(
        "Profit factor:",
        round(profit_factor, 3)
        if np.isfinite(profit_factor)
        else "inf"
    )
    print("Max drawdown:", round(max_drawdown, 2))

    print()
    print("BUY:")
    print("  Trades:", len(buy_trades))
    print("  Wins:", buy_wins)
    print("  Total R:", round(buy_R, 2))

    print()
    print("SELL:")
    print("  Trades:", len(sell_trades))
    print("  Wins:", sell_wins)
    print("  Total R:", round(sell_R, 2))

    # Spike analysis
    bins = [
        -np.inf, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, np.inf
    ]

    labels = [
        "<0.75", "0.75-1", "1-1.5", "1.5-2",
        "2-3", "3-4", ">4"
    ]

    trades_df["spike_bucket"] = pd.cut(
        trades_df["spike_body"],
        bins=bins,
        labels=labels
    )

    spike_stats = (
        trades_df
        .groupby("spike_bucket", observed=False)
        .agg(
            setups=("R", "count"),
            wins=("R", lambda x: int((x > 0).sum())),
            losses=("R", lambda x: int((x <= 0).sum())),
            total_R=("R", "sum")
        )
    )

    spike_stats["win_rate"] = (
        spike_stats["wins"]
        /
        spike_stats["setups"]
        *
        100
    )

    print()
    print("========================================")
    print("SPIKE SIZE VS RESULT")
    print("========================================")
    print(spike_stats.round(2))

    # SL distance analysis
    if len(trades_df) >= 4:

        try:
            trades_df["sl_bucket"] = pd.qcut(
                trades_df["sl_points"],
                q=4,
                duplicates="drop"
            )

            sl_stats = (
                trades_df
                .groupby("sl_bucket", observed=False)
                .agg(
                    setups=("R", "count"),
                    wins=("R", lambda x: int((x > 0).sum())),
                    losses=("R", lambda x: int((x <= 0).sum())),
                    total_R=("R", "sum")
                )
            )

            sl_stats["win_rate"] = (
                sl_stats["wins"]
                /
                sl_stats["setups"]
                *
                100
            )

            print()
            print("========================================")
            print("SL DISTANCE VS RESULT")
            print("========================================")
            print(sl_stats.round(2))

        except Exception as e:
            print("SL bucket analysis skipped:", e)

    print()
    print("========================================")
    print("EXIT REASONS")
    print("========================================")
    print(trades_df["exit_reason"].value_counts())

else:

    print()
    print("========================================")
    print("BACKTEST RESULT")
    print("========================================")
    print("No valid trades were produced.")


# ============================================================
# OBJECTS CREATED
# ============================================================

all_signals = df[
    df["position"] != 0
].copy()

signal_count = len(all_signals)

print()
print("========================================")
print("OBJECTS CREATED")
print("========================================")

print("df          -> complete detection dataframe")
print("all_signals -> dataframe rows marked at executed entry (not all candidate setups)")
print("trades_df   -> completed backtest trade legs")
print()
print("DONE.")


# ============================================================
# SETTINGS SUMMARY
# ============================================================

print()
print("========================================")
print("LIVE-ALIGNED EXECUTION SETTINGS")
print("========================================")
print("Live-aligned execution :", USE_LIVE_ALIGNED_EXECUTION)
print("Current extreme entry  :", USE_CURRENT_EXTREME_AS_ENTRY)
print("Same-candle entry option:", ALLOW_SAME_CANDLE_SETUP_ENTRY)
print("Same-candle entry model :", (
    "Tick: next 0.5s poll after setup detection"
    if TICK_DATA_ACTIVE and EXACT_LIVE_0_5S_STATE_MACHINE
    else "OHLC: unavailable; setup is detected at completed bar"
))
print("Replace pending setup  :", REPLACE_PENDING_ON_NEW_SETUP)
print("Trend includes entry   :", TREND_INCLUDES_ENTRY_CANDLE)
print("Filter candle mode     :", FILTER_CANDLE_MODE)
print("SL mode                :", SL_MODE)
print("LowerLow lookback      :", LOWER_LOW_LOOKBACK)
print("Tick-aware execution   :", TICK_DATA_ACTIVE)
print("MT5 tick auto fallback :", USE_MT5_TICK_DATA_IF_AVAILABLE)
print("MT5 tick batch size    :", MT5_TICK_BATCH_SIZE)
print("Entry 1 polling interval:", POLL_INTERVAL_SECONDS, "seconds")
print("Entry 1 polling phase   :", POLL_PHASE_SECONDS, "seconds")
print("Tick model             : Bid/Ask + chronological ticks")
print("Tick duplicates kept   :", TICK_DATA_ACTIVE)
print()
print("Note: Tick and OHLC execution engines are independent.")
print("When Tick execution is ACTIVE, no OHLC execution fallback is used.")
print("When Tick data is unavailable, the OHLC engine is used independently.")

